# Olala mini 7A1B GGUF — llama.cpp fork on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/olala-mini-7a1b-gguf-llamacpp-62dd/colab/Olala-mini-7A1B-GGUF-llamacpp-fork.ipynb)

[`OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF) is the Apache-2.0 GGUF of OVHai LLM's (OVHcloud) [`Olala-v1.0-mini-7A1B`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B). The [base card](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B) is a **text-only** post-trained chat / tools model: **6.8B total parameters, about 1B active per token**, custom architecture **`olala`**, **32K** context (input + output). The hybrid stack is **36 layers: 29 Mamba-3 and 7 attention** (one in five), each followed by a mixture-of-experts block with **256 routed experts (6 active) and 1 shared expert**. The repo is **not gated**. Licence: **Apache 2.0**.

**Why this model.** It is OVHcloud's first open language-model family ([announcement](https://x.com/OVHcloud/status/2123370331406483592)): a small reasoning model that thinks at four effort levels (`disabled`, `low`, `medium`, `high`) and emits tool calls as XML with JSON arguments. Stock engines cannot load `olala`. The GGUF card says upstream llama.cpp, Ollama, LM Studio, and the Hub "Use this model" apps do **not** support the architecture yet. The Transformers path wants `transformers==5.8.*` plus OVHai's Mamba fork and scattermoe. This notebook stays on the published GGUF and **builds their [llama.cpp fork](https://github.com/OVHai-LLM/llama.cpp)** with CUDA.

**Why a fork compile, not a wheel.** `llama-cpp-python` cannot load `olala`. The fork publishes **no GitHub Releases** (so no CUDA 12 / sm_75 Linux tarball to download). The GGUF card's recipe is clone `OVHai-LLM/llama.cpp`, `cmake -DGGML_CUDA=ON -DLLAMA_USE_PREBUILT_UI=OFF`, then `cmake --build`. This notebook still checks Releases at runtime; if none match Colab, it source-builds **only `llama-completion`** with **`-j 1`**, **`-DCMAKE_CUDA_ARCHITECTURES=75`**, and FlashAttention-all-quants off. A free T4 has 2 vCPUs and ~12.7 GB system RAM — two parallel `nvcc` jobs OOM the VM with no error text. The cell streams the log, prints wall-clock time, and prints free RAM before/after. Expect **about 20–40 minutes**.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about **15 GB** of VRAM. Default file: **`Olala-v1.0-mini-7A1B-q8_0-safe2.gguf` (6.9 GB)** from the GGUF card. **`q6_k` (6.3 GB)** then **`q5_k_m` (6.0 GB)** are fallbacks if this GPU OOMs or hits a kernel issue. Do **not** download the 13.6 GB BF16 file on a T4. Demos use **`-c 8192`** (card max 32,768; recurrent models cannot shift a prompt that overruns `-c`). All layers on GPU (`-ngl 99`). Sampler from the cards: **temperature 0.8, top_p 0.9, top_k 40**. Reasoning effort **`low`**, matching the GGUF card's NVIDIA example (template default is `medium`). Flash attention **on** (`-fa 1`) as the GGUF card states. If sm_75 rejects that kernel, the notebook retries `-fa 0`.

The GGUF chat template file is [`Olala-v1.0-mini-7A1B-chat-template.jinja`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF/blob/main/Olala-v1.0-mini-7A1B-chat-template.jinja). Pass `--jinja` **before** `--chat-template-file`.

This walkthrough is **inference only**. It does not train or fine-tune.

**Use cases:** [docs/Olala-mini-7A1B-GGUF-llamacpp-fork.md](https://github.com/donvito/notebooks/blob/melvin/olala-mini-7a1b-gguf-llamacpp-62dd/docs/Olala-mini-7A1B-GGUF-llamacpp-fork.md)

**Links:** [GGUF](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF) · [base card](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B) · [fork](https://github.com/OVHai-LLM/llama.cpp) · [announcement](https://x.com/OVHcloud/status/2123370331406483592)


## 1. Check the GPU

Confirm a T4 (or better) is attached **before** compiling the fork. The next cell prints `torch.cuda.is_available()`, the GPU name, compute capability, and free VRAM, then stops on CPU.


In [ ]:
import subprocess

import torch


def print_vram(label=""):
    # llama.cpp allocates CUDA memory outside PyTorch. nvidia-smi is the measured counter.
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")
    used_mib = int(smi.split(",")[0].strip().split()[0])
    return used_mib


def print_host_ram(label=""):
    info = {}
    with open("/proc/meminfo") as fh:
        for line in fh:
            key, value = line.split(":", 1)
            info[key] = value.strip()
    suffix = f" ({label})" if label else ""
    print(f"host MemTotal/MemAvailable{suffix}: {info.get('MemTotal')} / {info.get('MemAvailable')}")


print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)
name = torch.cuda.get_device_name()
total_gb = props.total_memory / 1024**3
free_b, total_b = torch.cuda.mem_get_info()
print()
print(f"GPU name: {name}")
print(f"compute capability: sm_{major}{minor}")
print(f"VRAM total: {total_gb:.1f} GB")
print(f"torch.cuda.mem_get_info free: {free_b / 1024**3:.1f} GB / {total_b / 1024**3:.1f} GB")
print_vram("before build")
print_host_ram("before build")
print()
print("=== T4-first notes ===")
print("- Architecture is `olala`. Upstream llama.cpp / llama-cpp-python cannot load this GGUF.")
print("- OVHai-LLM/llama.cpp has no GitHub CUDA Linux releases. This notebook source-builds llama-completion.")
print("- Build is -j 1, sm_75 only, FA-all-quants off. Expect about 20-40 minutes. Do not raise -j.")
print("- Default file: Olala-v1.0-mini-7A1B-q8_0-safe2.gguf (6.9 GB on the GGUF card).")
print("- Fallbacks if this GPU fails: q6_k (6.3 GB), q5_k_m (6.0 GB), then fewer -ngl.")
print("- Do not download Olala-v1.0-mini-7A1B-bf16.gguf (13.6 GB) on a 15 GB T4.")
print("- Card max context is 32,768. Demos use -c 8192. Recurrent models cannot shift overflow.")
print("- Sampling from the cards: temperature 0.8, top_p 0.9, top_k 40. reasoning_effort=low.")
print("- -fa 1 as the GGUF card. Retry -fa 0 if a Turing kernel errors.")
print("- The fork's older CUDA notes warn quantized files can NaN on GPU; the published GGUF card")
print("  says all four files run on NVIDIA. If you see NaN / ?????, try q6_k then q5_k_m.")
if major < 8:
    print("- This GPU is below sm_80 (no native BF16, no FA2). The path is GGUF + llama.cpp -fa.")
PEAK_VRAM_MIB = print_vram("after gpu check")


## 2. Build the OVHai-LLM llama.cpp fork (CUDA, sm_75)

The GGUF card's CUDA configure line is:

```sh
cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_USE_PREBUILT_UI=OFF
```

This cell first asks GitHub Releases for a CUDA 12 / sm_75 Linux binary. The fork currently has **none**, so it source-builds. T4-only cmake flags, checked against this fork's `CMakeLists.txt`:

- `-DCMAKE_CUDA_ARCHITECTURES=75` — Turing only (do not let nvcc emit every arch)
- `-DGGML_CUDA_FA_ALL_QUANTS=OFF` — skip the full FlashAttention quant matrix (already the default; set explicitly)
- `-DGGML_NATIVE=OFF`, `-DLLAMA_BUILD_TESTS=OFF`, `-DLLAMA_BUILD_EXAMPLES=OFF`, `-DLLAMA_BUILD_SERVER=OFF`
- `-DGGML_CUDA_NCCL=OFF`, `-DLLAMA_OPENSSL=OFF`
- `CMAKE_CUDA_FLAGS=--threads=1` so one `nvcc` does not fan out extra host threads

`LLAMA_CURL` is deprecated in this fork (ignored). There is no `GGML_CUDA_F16` option and no flag to skip unused quant MMQ kernels — those `.cu` files are always globbed.

**One target, one job.** In this fork `llama-cli` links `llama-server-impl` and is only added when `LLAMA_BUILD_SERVER` is on. Demos use **`llama-completion`** (`--jinja --single-turn`), which does not pull in the server. `cmake --build … -j 1 --target llama-completion`. Parallel `-j 2` froze a free T4 (~12.7 GB RAM) about 3–4 minutes in with no error text.

The log is streamed to this cell (not captured silently). Wall-clock time and `/proc/meminfo` MemAvailable print before and after. Re-running skips the compile when `llama-completion` is already in `build/bin`. Expect **about 20–40 minutes**.


In [ ]:
%pip install -qqq --no-cache-dir huggingface_hub

import json
import os
import shutil
import subprocess
import tarfile
import time
import urllib.error
import urllib.request
import zipfile

ROOT = "/content" if os.path.isdir("/content") else "/tmp"
SRC = os.path.join(ROOT, "llama.cpp-olala")
BUILD = os.path.join(SRC, "build")
FORK = "https://github.com/OVHai-LLM/llama.cpp.git"
RELEASES_API = "https://api.github.com/repos/OVHai-LLM/llama.cpp/releases?per_page=10"
CLI = os.path.join(BUILD, "bin", "llama-completion")

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "1"
if os.path.isdir("/usr/local/cuda"):
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

if shutil.which("cmake") is None or shutil.which("g++") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "cmake", "build-essential", "git"],
        check=True,
    )

print("cmake:", shutil.which("cmake"))
print("g++:", shutil.which("g++"))
print("nvcc:", shutil.which("nvcc"))
print("nproc:", os.cpu_count())
if shutil.which("nvcc"):
    subprocess.run(["nvcc", "--version"], check=False)


def find_extracted_binary(root, names=("llama-completion", "llama-cli")):
    for dirpath, _, filenames in os.walk(root):
        for name in names:
            path = os.path.join(dirpath, name)
            if os.path.isfile(path) and os.access(path, os.X_OK):
                return path
    return None


def asset_score(name: str) -> int:
    n = name.lower()
    if any(tok in n for tok in ("win", "macos", "darwin", "musl", "android")):
        return 0
    if not (("linux" in n) or ("ubuntu" in n)):
        return 0
    if "cuda" not in n:
        return 0
    score = 1
    if "sm_75" in n or "sm75" in n:
        score += 10
    if any(tok in n for tok in ("cuda-12", "cuda12", "12.8", "12.4", "12.6")):
        score += 5
    return score


def try_prebuilt_cuda_binary() -> bool:
    print("checking OVHai-LLM/llama.cpp GitHub Releases for a CUDA Linux / sm_75 binary...")
    req = urllib.request.Request(
        RELEASES_API,
        headers={"User-Agent": "donvito-notebooks/olala-colab", "Accept": "application/vnd.github+json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            releases = json.loads(resp.read().decode())
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        print("release check failed:", exc)
        return False
    if not releases:
        print("no GitHub releases published; source-building llama-completion")
        return False
    best = None
    best_score = 0
    for rel in releases:
        for asset in rel.get("assets") or []:
            name = asset.get("name") or ""
            url = asset.get("browser_download_url") or ""
            score = asset_score(name)
            if url and score > best_score:
                best = (name, url, rel.get("tag_name"))
                best_score = score
    if not best:
        print("releases exist but none look like CUDA Linux Colab binaries; source-building")
        return False
    name, url, tag = best
    print(f"trying prebuilt {name} (tag {tag})")
    archive = os.path.join(ROOT, name)
    urllib.request.urlretrieve(url, archive)
    extract_dir = os.path.join(ROOT, "olala-prebuilt")
    shutil.rmtree(extract_dir, ignore_errors=True)
    os.makedirs(extract_dir, exist_ok=True)
    if name.endswith(".zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(extract_dir)
    else:
        with tarfile.open(archive) as tf:
            tf.extractall(extract_dir)
    found = find_extracted_binary(extract_dir)
    if not found:
        print("archive had no llama-completion / llama-cli; source-building")
        return False
    dest = CLI
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(found, dest)
    os.chmod(dest, 0o755)
    print("installed prebuilt binary:", dest)
    return True


def stream_cmd(cmd, cwd, log_path):
    wrapped = cmd
    if shutil.which("stdbuf"):
        wrapped = ["stdbuf", "-oL", "-eL", *cmd]
    print("exec:", " ".join(cmd), flush=True)
    proc = subprocess.Popen(
        wrapped,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    with open(log_path, "a") as log_f:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log_f.write(line)
    return proc.wait()


if not os.path.isdir(os.path.join(SRC, ".git")):
    print("cloning", FORK)
    subprocess.run(["git", "clone", "--depth", "1", FORK, SRC], check=True)
else:
    print("reuse clone", SRC)

subprocess.run(["git", "-C", SRC, "log", "-1", "--oneline"], check=False)

print_host_ram("before fork binary")
used_prebuilt = False
if os.path.isfile(CLI):
    print("already built:", CLI)
else:
    used_prebuilt = try_prebuilt_cuda_binary()
    if not used_prebuilt:
        cmake_cmd = [
            "cmake", "-B", BUILD,
            "-DGGML_CUDA=ON",
            "-DCMAKE_BUILD_TYPE=Release",
            "-DLLAMA_USE_PREBUILT_UI=OFF",
            "-DCMAKE_CUDA_ARCHITECTURES=75",
            "-DGGML_CUDA_FA_ALL_QUANTS=OFF",
            "-DGGML_NATIVE=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF",
            "-DLLAMA_BUILD_SERVER=OFF",
            "-DGGML_CUDA_NCCL=OFF",
            "-DLLAMA_OPENSSL=OFF",
            "-DCMAKE_CUDA_FLAGS=--threads=1",
        ]
        nvcc = shutil.which("nvcc")
        if nvcc:
            cmake_cmd.append(f"-DCMAKE_CUDA_COMPILER={nvcc}")
        log_path = os.path.join(ROOT, "olala-llama-build.log")
        if os.path.isfile(log_path):
            os.remove(log_path)
        print("configure:", " ".join(cmake_cmd), flush=True)
        print("log:", log_path, flush=True)
        t0 = time.time()
        cfg_rc = stream_cmd(cmake_cmd, SRC, log_path)
        if cfg_rc != 0:
            raise SystemExit(f"cmake configure failed with exit {cfg_rc}")
        build_cmd = ["cmake", "--build", BUILD, "-j", "1", "--target", "llama-completion"]
        print("build:", " ".join(build_cmd), flush=True)
        bld_rc = stream_cmd(build_cmd, SRC, log_path)
        elapsed = time.time() - t0
        print(
            f"cmake wall time: {elapsed:.1f} s ({elapsed / 60:.1f} min)  jobs=1  target=llama-completion  arch=75",
            flush=True,
        )
        if bld_rc != 0:
            raise SystemExit(f"fork build failed with exit {bld_rc}")

assert os.path.isfile(CLI), CLI
print("llama-completion:", CLI)
print("used_prebuilt:", used_prebuilt if os.path.isfile(CLI) else False)
print_host_ram("after fork binary")
print_vram("after build")


## 3. Download the GGUF and the card's chat template

Default file: **`Olala-v1.0-mini-7A1B-q8_0-safe2.gguf`**, **6.9 GB** on the [GGUF files table](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF). Also download **`Olala-v1.0-mini-7A1B-chat-template.jinja`**. To try a smaller quant on the same T4, set `FILENAME` to `Olala-v1.0-mini-7A1B-q6_k.gguf` (6.3 GB) or `Olala-v1.0-mini-7A1B-q5_k_m.gguf` (6.0 GB).

`HF_HUB_DISABLE_IMPLICIT_TOKEN=1` — the repo is public. Cancel any Colab `HF_TOKEN` prompt.


In [ ]:
import os
import re

from huggingface_hub import hf_hub_download

REPO = "OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF"
FILENAME = "Olala-v1.0-mini-7A1B-q8_0-safe2.gguf"
TEMPLATE_NAME = "Olala-v1.0-mini-7A1B-chat-template.jinja"
FALLBACK_FILES = [
    "Olala-v1.0-mini-7A1B-q8_0-safe2.gguf",  # 6.9 GB
    "Olala-v1.0-mini-7A1B-q6_k.gguf",        # 6.3 GB
    "Olala-v1.0-mini-7A1B-q5_k_m.gguf",      # 6.0 GB
]
N_CTX = 8192
NGL = "99"
FA = "1"
SAMPLER = dict(temperature=0.8, top_p=0.9, top_k=40)
REASONING = "low"

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

TEMPLATE = hf_hub_download(REPO, TEMPLATE_NAME)
MODEL = hf_hub_download(REPO, FILENAME)
print("template", TEMPLATE)
print("model", MODEL, "size_gb", round(os.path.getsize(MODEL) / 1024**3, 2))
print_vram("after download (weights still on disk)")


def record_vram(label):
    global PEAK_VRAM_MIB
    used = print_vram(label)
    PEAK_VRAM_MIB = max(PEAK_VRAM_MIB, used)
    return used


def looks_like_kernel_or_nan(text: str) -> bool:
    t = text.lower()
    needles = (
        "cuda_error", "invalid configuration argument", "no kernel image",
        "?????", "out of memory", "failed to allocate",
        "cuda malloc", "illegal memory", "unspecified launch failure",
    )
    if any(n in t for n in needles):
        return True
    return re.search(r"\bnan\b", t) is not None


def parse_prompt_tokens(text: str):
    m = re.search(r"prompt eval time.*?/\s*(\d+)\s*tokens", text)
    return int(m.group(1)) if m else None


## 4. Demo 1 — short chat + tokens/s (`llama-completion`)

One user turn, then the process exits (`-st` / `--single-turn`). Flags match the GGUF card's NVIDIA line: `-ngl 99 -fa 1 --temp 0.8 --top-p 0.9 --top-k 40 --reasoning-effort low`, plus `--jinja --chat-template-file` and `-n 192`. `llama-completion` prints **prompt eval** and **eval** tokens/s at the end (`llama_perf_context_print`). Those are this run's numbers, not the Mac/H100 tables on the cards.

This fork's `llama-cli` target links the server library, so the notebook builds `llama-completion` instead (same common flags, no `llama-server`).

If the first attempt fails with a CUDA kernel / OOM / NaN, the cell retries `-fa 0`, then the next smaller GGUF, then `-ngl 24`.


In [ ]:
import subprocess
import time

CHAT_PROMPT = (
    "In three short sentences, what is a heat pump, and why can it heat a room "
    "while using less electrical energy than a resistive heater?"
)


def completion_cmd(model, ngl, fa, prompt, n_predict, extra_args=None, sys_prompt=None, prompt_file=None):
    cmd = [
        CLI,
        "-m", model,
        "-ngl", str(ngl),
        "-fa", str(fa),
        "-c", str(N_CTX),
        "--jinja",
        "--chat-template-file", TEMPLATE,
        "--reasoning-effort", REASONING,
        "--temp", str(SAMPLER["temperature"]),
        "--top-p", str(SAMPLER["top_p"]),
        "--top-k", str(SAMPLER["top_k"]),
        "-st",
        "-n", str(n_predict),
    ]
    if sys_prompt:
        cmd.extend(["-sys", sys_prompt])
    if prompt_file:
        cmd.extend(["-f", prompt_file])
    else:
        cmd.extend(["-p", prompt])
    if extra_args:
        cmd.extend(extra_args)
    return cmd


def run_completion(model, ngl, fa, prompt, n_predict=192, extra_args=None, sys_prompt=None,
                   prompt_file=None, timeout=300):
    cmd = completion_cmd(model, ngl, fa, prompt, n_predict, extra_args, sys_prompt, prompt_file)
    print("exec:", " ".join(cmd[:12]), "...")
    t0 = time.time()
    try:
        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            stdin=subprocess.DEVNULL,
            timeout=timeout,
        )
        elapsed = time.time() - t0
        return proc.returncode, proc.stdout, elapsed
    except subprocess.TimeoutExpired as exc:
        elapsed = time.time() - t0
        out = exc.stdout or ""
        if isinstance(out, bytes):
            out = out.decode("utf-8", "replace")
        return 124, out + f"\n[timeout after {timeout}s]", elapsed


def download_quant(name):
    path = hf_hub_download(REPO, name)
    print("downloaded fallback", name, "size_gb", round(os.path.getsize(path) / 1024**3, 2))
    return path


plans = [
    dict(file=FILENAME, ngl=99, fa=1, note="card default: q8_0-safe2, all layers, -fa 1"),
    dict(file=FILENAME, ngl=99, fa=0, note="same file, -fa 0 (Turing kernel fallback)"),
    dict(file="Olala-v1.0-mini-7A1B-q6_k.gguf", ngl=99, fa=1, note="q6_k 6.3 GB, -fa 1"),
    dict(file="Olala-v1.0-mini-7A1B-q5_k_m.gguf", ngl=99, fa=1, note="q5_k_m 6.0 GB, -fa 1"),
    dict(file="Olala-v1.0-mini-7A1B-q5_k_m.gguf", ngl=24, fa=0, note="q5_k_m, 24 GPU layers, -fa 0"),
]

used_plan = None
cli_out = ""
for plan in plans:
    if plan["file"] != os.path.basename(MODEL):
        MODEL = download_quant(plan["file"])
        FILENAME = plan["file"]
    NGL, FA = str(plan["ngl"]), str(plan["fa"])
    print()
    print("=== try:", plan["note"], "===")
    code, cli_out, elapsed = run_completion(MODEL, plan["ngl"], plan["fa"], CHAT_PROMPT, n_predict=192)
    print(cli_out[-6000:] if len(cli_out) > 6000 else cli_out)
    print(f"llama-completion exit {code} in {elapsed:.1f}s")
    record_vram("after llama-completion")
    if code == 0 and not looks_like_kernel_or_nan(cli_out):
        used_plan = plan
        break
    print("that attempt failed; next fallback")
else:
    raise SystemExit(
        "llama-completion did not produce a clean run on this GPU. "
        "See the log above (kernel / OOM / NaN). The 13.6 GB BF16 file does not fit a T4."
    )

print()
print("using", used_plan)
for line in cli_out.splitlines():
    if "tokens per second" in line or "llama_perf" in line:
        print("perf:", line)


## 5. Demo 2 — tool call / JSON function call (`llama-completion`)

The fork [setup guide](https://github.com/OVHai-LLM/llama.cpp/blob/main/docs/olala-setup.md) and the base card use **`get_weather`** with a `city` argument. Generated calls are XML with JSON arguments in the `tools` channel:

```xml
<toolcalls><call id="call_0"><name>get_weather</name><arguments>{"city": "Paris"}</arguments></call></toolcalls>
```

`llama-server` is not built on this T4 (it is a large extra target and this fork's `llama-cli` links it). The cell asks `llama-completion` for the XML call, runs a **mocked** weather function, and sends the tool result back in a second `-st` turn. No real weather API.


In [ ]:
import json

WEATHER = {
    "Paris": {"temp_c": 12, "conditions": "light rain", "wind_kmh": 18},
    "Roubaix": {"temp_c": 9, "conditions": "overcast", "wind_kmh": 22},
}

TOOL_SYS = (
    "You have one tool: get_weather. It takes a JSON object with a required string field "
    '"city". When you need the weather, emit XML exactly like '
    '<toolcalls><call id="call_0"><name>get_weather</name><arguments>{"city": "Paris"}'
    "</arguments></call></toolcalls> "
    "and do not invent temperatures. After you receive a tool result, answer in one sentence."
)


def parse_tool_calls(blob: str):
    found = []
    for m in re.finditer(
        r'<call id="([^"]+)"><name>([^<]+)</name><arguments>(.*?)</arguments></call>',
        blob,
        flags=re.S,
    ):
        found.append(
            {
                "id": m.group(1),
                "name": m.group(2),
                "arguments": m.group(3).strip(),
            }
        )
    return found


print("=== user ===")
print("What is the weather in Paris?")
print()
print("=== first assistant turn (expect a get_weather tool call) ===")
code, first_out, elapsed = run_completion(
    MODEL, NGL, FA, "What is the weather in Paris?", n_predict=256, sys_prompt=TOOL_SYS
)
print(first_out[-6000:] if len(first_out) > 6000 else first_out)
print(f"llama-completion exit {code} in {elapsed:.1f}s")
record_vram("after tool-call turn")
if code != 0 or looks_like_kernel_or_nan(first_out):
    raise SystemExit("tool-call turn failed; see the log above.")

calls = parse_tool_calls(first_out)
if not calls:
    raise SystemExit("No <toolcalls> XML. The model answered in text; see the dump above.")
print("parsed tool_calls:", json.dumps(calls, indent=2)[:2000])

tool_lines = []
for call in calls:
    raw_args = call.get("arguments") or "{}"
    args = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
    city = (args or {}).get("city", "Paris")
    name = call.get("name")
    if name != "get_weather":
        result = {"error": f"unknown tool {name}"}
    else:
        result = {"city": city, **WEATHER.get(city, {"temp_c": None, "conditions": "unknown", "note": "mocked"})}
    print()
    print("mocked", name, args, "->", result)
    tool_lines.append(f"Tool {name} returned: {json.dumps(result)}")

follow_up = (
    "You called get_weather for the user question 'What is the weather in Paris?'.\n"
    + "\n".join(tool_lines)
    + "\nGive a one-sentence answer using only that tool result."
)
print()
print("=== second turn (tool result in context) ===")
code, final_out, elapsed = run_completion(
    MODEL, NGL, FA, follow_up, n_predict=192, sys_prompt=TOOL_SYS
)
print(final_out[-4000:] if len(final_out) > 4000 else final_out)
print(f"llama-completion exit {code} in {elapsed:.1f}s")
record_vram("after tool-result turn")


## 6. Demo 3 — few-thousand-token summarisation

The cards give a **32K** window. This cell writes a synthetic operations brief a few thousand tokens long, asks `llama-completion` for an eight-bullet summary (`-f` so the prompt is not on the argv), and keeps `-c 8192` so the prompt fits. Prompts longer than `-c` fail: recurrent models cannot shift context. Prompt size is read from `llama_perf` (`prompt eval … / N tokens`).


In [ ]:
def make_brief() -> str:
    header = """NORTHWIND FULFILMENT — WEEKLY OPERATIONS BRIEF
Site: Roubaix DC-3 (mock document for a context-window demo; not real OVHcloud data)
Week of 10–16 November 2026
Audience: site lead, inbound, inventory, outbound, transport, people, safety
Please summarise for a Monday standup. Do not invent sites, vendors, or numbers that are not in this brief.

"""
    streams = [
        (
            "Inbound docks",
            "Dock 1 handled live trailers from Lille and Ghent. Average unload was 41 minutes against a 35-minute target. "
            "Two trailers waited in the yard because the ASN barcodes used last year's vendor prefix. "
            "Dock 2 stayed on the parcel belt. A jam at 14:10 stopped the belt for 18 minutes; maintenance replaced a seized roller. "
            "Dock 3 is still closed for epoxy repair, now due Friday. Overflow used Dock 4 and added about 12 minutes of tug travel per load.",
        ),
        (
            "Inventory accuracy",
            "Cycle counts on aisles R12–R18 finished at 98.4% match to WMS, down from 99.1% last week. "
            "The misses cluster on mixed totes that the putaway RF gun still splits into two licence plates. "
            "A patch is scheduled Tuesday 02:00–04:00. Until then, pickers must scan both plates before leaving the aisle. "
            "Freeze on bulk candy in R04 remains: a temperature logger drifted 1.8 C overnight Thursday.",
        ),
        (
            "Outbound and carrier mix",
            "Orders shipped: 18,440, 4% above plan. Same-day cutoff was pulled forward 20 minutes on Thursday when a tractor failed inspection. "
            "Carrier mix: 61% parcel, 27% groupage, 12% dedicated. Groupage on-time dropped to 91% because the 19:00 trunk to Paris left 35 minutes late. "
            "Customer 'Atlas Kitchen' asked to split pallets by store from Wednesday; that adds one extra label per pallet and about 40 seconds.",
        ),
        (
            "People and safety",
            "Headcount: 212 on payroll, 28 agency. Two agency forklift drivers finished site induction Monday. "
            "Near miss Wednesday: a pallet jack rolled toward the empty-pallet press; the gate interlock held. Action: yellow floor marks and a toolbox talk Friday. "
            "Overtime was 6.2% of hours, mostly outbound Thursday. Flu absences peaked at 9 people on Friday morning.",
        ),
        (
            "Systems",
            "WMS build 2026.11.2 landed Saturday and cleared the duplicate-LPN bug from 3 November. "
            "Label printers 4 and 7 still drop Wi-Fi around 16:00 when the guest network fills; IT will VLAN them this week. "
            "The yard camera on gate B lost focus after rain; a replacement dome arrives Wednesday.",
        ),
        (
            "Next week",
            "Monday: fire-door test at 06:30, expect 12 minutes with no inbound. "
            "Tuesday: WMS putaway patch. Wednesday: Atlas Kitchen store-split labels go live on lane 2 only. "
            "Thursday: peak staffing plan for Black Friday week (draft in the shared folder). "
            "Friday: Dock 3 epoxy inspection. Do not book live unload on Dock 3 until quality signs off.",
        ),
    ]
    parts = [header]
    for i, (title, body) in enumerate(streams, 1):
        extras = "\n".join(
            f"- Checkpoint {j}: {title} remains on the standup list until the owner records a close date."
            for j in range(1, 12)
        )
        follow = (
            f"Owner notes for {title}: keep the same metric definitions as last quarter. "
            f"Do not change the SLA without writing it in this brief. "
            f"If a number is missing, say it is missing. "
        ) * 8
        parts.append(f"## {i}. {title}\n\n{body}\n\n{follow}\n\n{extras}")
    # Extra unique lines so the prompt sits in the few-thousand-token range.
    for n in range(1, 41):
        parts.append(
            f"Appendix line {n:02d}. This paragraph exists so the summarisation prompt is a few thousand "
            f"tokens, not a tweet. It restates that Dock 3 is closed, cycle-count accuracy is 98.4%, "
            f"orders shipped were 18,440, groupage on-time is 91%, overtime is 6.2%, and the WMS patch "
            f"is Tuesday 02:00–04:00. No other sites or figures are introduced here."
        )
    return "\n\n".join(parts)


brief = make_brief()
approx = max(1, len(brief) // 4)
print(f"brief characters: {len(brief):,}  rough token estimate (chars/4): {approx:,}")
print("first 400 chars:\n", brief[:400], "\n...")

sum_prompt_path = os.path.join(ROOT, "olala-brief-prompt.txt")
with open(sum_prompt_path, "w") as fh:
    fh.write(
        "Summarise the operations brief below in exactly eight short bullets for a Monday standup. "
        "Cover inbound, inventory, outbound, people/safety, systems, and next week. "
        "Use only numbers that appear in the brief. After the bullets, add one 'watch-outs' line.\n\n"
        + brief
    )
print("wrote", sum_prompt_path, "bytes", os.path.getsize(sum_prompt_path))

print()
print("=== summarisation ===")
code, sum_out, elapsed = run_completion(
    MODEL, NGL, FA, prompt="", n_predict=400, prompt_file=sum_prompt_path, timeout=600,
)
print(sum_out[-8000:] if len(sum_out) > 8000 else sum_out)
print(f"llama-completion exit {code} in {elapsed:.1f}s")
record_vram("after summarisation")
prompt_tokens = parse_prompt_tokens(sum_out)
print("prompt_tokens (llama_perf):", prompt_tokens)
if prompt_tokens is not None and prompt_tokens < 1500:
    print("NOTE: prompt_tokens came in under 1,500; the brief generator may need a longer pad on this tokenizer.")


## 7. Peak VRAM, caveats, cleanup

`nvidia-smi` used memory is the peak tracker (`PEAK_VRAM_MIB`). llama.cpp does not allocate through `torch.cuda`.

**Card facts this notebook relies on (not T4 measurements):** 6.8B total / ~1B active; 32K context; Apache-2.0; `olala`; GGUF sizes 13.6 / 6.9 / 6.3 / 6.0 GB; sampling 0.8 / 0.9 / 40; fork required. Mac and H100 tokens/s on the cards are **not** this T4 run.

**T4 risks.** CUDA compile of `llama-completion` with `-j 1` takes about **20–40 minutes** on a free Colab T4. Parallel nvcc (`-j 2`) can freeze the VM with no error (RAM/CPU starvation). sm_75 kernel gaps. Quantized-on-GPU NaN warned in the fork's older setup notes. 6.9 GB weights plus 8K context on 15 GB. Recurrence: do not send a prompt longer than `-c`.

The GPU is free as soon as `llama-completion` exits. Open another notebook after this cell if you want the session for something else:


In [ ]:
print(f"peak nvidia-smi used (this session): {PEAK_VRAM_MIB} MiB  ({PEAK_VRAM_MIB / 1024:.2f} GB)")
print("file", FILENAME, "| -ngl", NGL, "| -fa", FA, "| -c", N_CTX)
print("binary", CLI)
print_host_ram("end of session")
record_vram("end of session")
print()
print("Fork:", FORK)
print("GGUF:", f"https://huggingface.co/{REPO}")
print("Template:", TEMPLATE_NAME)
